In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parents[0] if Path.cwd(
).name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from utils.print_section import print_section

# ============================================================
# Load data
# ============================================================

INPUT_PATH = "../data/processed/model_features.csv"

df = pd.read_csv(
    INPUT_PATH,
    low_memory=False,
)

print(df.shape)

# ============================================================
# Features
# ============================================================

FEATURES = [
    # Financial variables
    "profitability",
    "liquidity",
    "solvency",
    "structure",
    "log_age",
    "size",

    # Behavioral variables
    "laat_dummy",
    "boete_dummy",
    "dgrmovestreet",
]

TARGET = "target"

X = df[FEATURES]
y = df[TARGET]

# ============================================================
# Train-test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# ============================================================
# Class imbalance
# ============================================================

negative_class = (y_train == 0).sum()
positive_class = (y_train == 1).sum()

scale_pos_weight = (
    negative_class / positive_class
)

print_section("Class imbalance")

print(f"Negative class: {negative_class:,}")
print(f"Positive class: {positive_class:,}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# ============================================================
# Model
# ============================================================

pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        XGBClassifier(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            eval_metric="logloss",
        )
    )
])

# ============================================================
# Fit
# ============================================================

pipeline.fit(
    X_train,
    y_train,
)

# ============================================================
# Predict
# ============================================================

y_pred = pipeline.predict(X_test)

y_prob = pipeline.predict_proba(X_test)[:, 1]

# ============================================================
# Performance
# ============================================================

print_section("Performance")

print(
    f"Accuracy : {accuracy_score(y_test, y_pred):.4f}"
)

print(
    f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}"
)

print(
    f"Recall   : {recall_score(y_test, y_pred):.4f}"
)

print(
    f"F1-score : {f1_score(y_test, y_pred):.4f}"
)

print(
    f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}"
)

# ============================================================
# Confusion matrix
# ============================================================

print_section("Confusion matrix")

cm = confusion_matrix(
    y_test,
    y_pred,
)

print(cm)

tn, fp, fn, tp = cm.ravel()

print()

print(f"True Negatives : {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives : {tp:,}")

# ============================================================
# Classification report
# ============================================================

print_section("Classification report")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0,
    )
)

# ============================================================
# Feature importance
# ============================================================

print_section("Feature importance")

importances = pd.DataFrame({
    "feature": FEATURES,
    "importance": (
        pipeline
        .named_steps["model"]
        .feature_importances_
    )
})

importances = importances.sort_values(
    by="importance",
    ascending=False,
)

print(importances)

# ============================================================
# Baseline
# ============================================================

print_section("Baseline")

print(
    f"Failure rate: {y.mean():.4%}"
)

# ============================================================
# Improvement over baseline
# ============================================================

print_section("Model improvement over baseline")

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0,
)

print(
    f"Baseline failure rate: "
    f"{y.mean():.4%}"
)

print(
    f"Model precision: "
    f"{precision:.4%}"
)

print(
    f"Improvement factor: "
    f"{precision / y.mean():.2f}x"
)

(981818, 96)

Class imbalance
Negative class: 783,196
Positive class: 2,258
scale_pos_weight: 346.85

Performance
Accuracy : 0.8208
Precision: 0.0111
Recall   : 0.6950
F1-score : 0.0218
ROC-AUC  : 0.8458

Confusion matrix
[[160775  35025]
 [   172    392]]

True Negatives : 160,775
False Positives: 35,025
False Negatives: 172
True Positives : 392

Classification report
              precision    recall  f1-score   support

           0       1.00      0.82      0.90    195800
           1       0.01      0.70      0.02       564

    accuracy                           0.82    196364
   macro avg       0.50      0.76      0.46    196364
weighted avg       1.00      0.82      0.90    196364


Feature importance
         feature  importance
2       solvency    0.239584
8  dgrmovestreet    0.223759
0  profitability    0.126288
3      structure    0.097535
5           size    0.081876
1      liquidity    0.081532
4        log_age    0.052369
6     laat_dummy    0.049664
7    boete_dummy    